In [6]:
import sys
from pathlib import Path
import pickle
import time
import logging
import numpy as np
from scipy.sparse import csr_matrix, save_npz
from sklearn.feature_extraction.text import CountVectorizer
from llama_index.core import Document

# Add parent directory to Python path
# Assuming your project root is `/work`, add it to the system path
project_root = Path('/work')
sys.path.append(str(project_root))
from src.utils.data_utils import load_documents

# Helper function to extract text from documents
def extract_texts(documents):
    return [doc.text if isinstance(doc, Document) else doc for doc in documents]

# Function to save index components.
def save_index(index_path: Path, vectorizer: CountVectorizer, embeddings, documents):
    index_path.mkdir(parents=True, exist_ok=True)
    
    with open(index_path / "vectorizer.pkl", "wb") as f:
        pickle.dump(vectorizer, f)
    
    save_npz(index_path / "embeddings.npz", embeddings)
    
    with open(index_path / "documents.pkl", "wb") as f:
        pickle.dump(documents, f)
    
    logging.info(f"Index saved to {index_path}")

# Function to load restricted vocabulary from file
def load_vocabulary(vocab_path):
    """Load vocabulary dictionary from pickle file"""
    try:
        with open(vocab_path, 'rb') as f:
            vocab_dict = pickle.load(f)
        print(f"Loaded vocabulary with {len(vocab_dict)} terms from {vocab_path}")
        return vocab_dict
    except Exception as e:
        print(f"Error loading vocabulary from {vocab_path}: {e}")
        return None

# Main function with progress updates and estimated time remaining.
def main(file, restricted_vocab_path=None, output_suffix=""):
    start = time.time()
    print("Starting Bag-of-Words indexing process...")

    # Handle restricted vocabulary
    allowed_vocab_dict = None
    if restricted_vocab_path:
        allowed_vocab_dict = load_vocabulary(restricted_vocab_path)
        if allowed_vocab_dict is None:
            print("Failed to load vocabulary. Proceeding with unrestricted vocabulary.")
        else:
            print(f"Using restricted vocabulary with {len(allowed_vocab_dict)} terms")
    else:
        print("Using unrestricted vocabulary")
    
    print("Setting up file paths...")
    documents_path_texto = Path('/work/data/processed/' + file + '_texto.pkl')
    documents_path_resumen = Path('/work/data/processed/' + file + '_resumen.pkl')

    # Add suffix to index paths if using restricted vocabulary
    base_suffix = output_suffix if output_suffix else ("_Restricted" if allowed_vocab_dict else "")    
    index_path_texto = Path(f'/work/sandbox/bow/indexes/texto{base_suffix}' )
    index_path_resumen = Path(f'/work/sandbox/bow/indexes/resumen{base_suffix}')

    print("Loading documents for 'texto'...")
    documents_texto = load_documents(documents_path_texto)
    print("Loading documents for 'resumen'...")
    documents_resumen = load_documents(documents_path_resumen)

    if documents_texto is None or documents_resumen is None:
        logging.error("Failed to load one or both document sets. Exiting.")
        return

    datasets = [
        ("texto", documents_texto, index_path_texto),
        ("resumen", documents_resumen, index_path_resumen)
    ]
    total_steps = len(datasets)

    for i, (name, docs, index_path) in enumerate(datasets, 1):
        print(f"\nProcessing {name} documents ({i}/{total_steps})...")
        step_start = time.time()
        
        print("Fitting Bag-of-Words vectorizer...")

        # Define vectorizer parameters
        vectorizer_params = {
            'binary': False,       # Count frequencies (set to True for binary BoW)
            'lowercase': True, 
            'token_pattern': r'\b\w+\b',
            # Optional BoW parameters you can tune:
            # 'max_features': None,     # Limit vocabulary size
            # 'min_df': 1,              # Ignore terms that appear in < min_df documents
            # 'max_df': 1.0,            # Ignore terms that appear in > max_df fraction of documents
            # 'ngram_range': (1, 1),    # Use (1, 2) for unigrams + bigrams
            # 'stop_words': None,       # Set to 'english' to remove common words
        }

        # *Add vocabulary parameter if restricted vocabulary is provided
        if allowed_vocab_dict:
            vectorizer_params['vocabulary'] = allowed_vocab_dict
            print(f"  Using restricted vocabulary with {len(allowed_vocab_dict)} terms")
        else:
            print("  Using unrestricted vocabulary")
        
        vectorizer = CountVectorizer(**vectorizer_params)
        texts = extract_texts(docs)
        vectorizer.fit(texts)
        
        print("Transforming documents to Bag-of-Words embeddings...")
        embeddings = vectorizer.transform(texts)

        print("Saving index for", name, "documents...")
        save_index(index_path, vectorizer, embeddings, docs)

        step_end = time.time()
        step_elapsed = step_end - step_start
        remaining_steps = total_steps - i
        estimated_remaining = step_elapsed * remaining_steps if remaining_steps > 0 else 0
        minutes, seconds = divmod(estimated_remaining, 60)
        print(f"Completed processing {name} documents in {step_elapsed:.2f} seconds. "
              f"Estimated time remaining: {int(minutes)} minutes, {int(seconds)} seconds.")

    end = time.time()
    total_time = end - start
    print(f"\nTotal execution time: {total_time:.2f} seconds")

In [7]:
# With custom output suffix
vocab_path = Path('/work/sandbox/bow/vocab_dict.pkl')
main('OEB', restricted_vocab_path=vocab_path, output_suffix="")

Starting Bag-of-Words indexing process...
Loaded vocabulary with 153 terms from /work/sandbox/bow/vocab_dict.pkl
Using restricted vocabulary with 153 terms
Setting up file paths...
Loading documents for 'texto'...
Loading documents for 'resumen'...

Processing texto documents (1/2)...
Fitting Bag-of-Words vectorizer...
  Using restricted vocabulary with 153 terms
Transforming documents to Bag-of-Words embeddings...
Saving index for texto documents...
Completed processing texto documents in 3.65 seconds. Estimated time remaining: 0 minutes, 3 seconds.

Processing resumen documents (2/2)...
Fitting Bag-of-Words vectorizer...
  Using restricted vocabulary with 153 terms
Transforming documents to Bag-of-Words embeddings...
Saving index for resumen documents...
Completed processing resumen documents in 1.99 seconds. Estimated time remaining: 0 minutes, 0 seconds.

Total execution time: 9.80 seconds
